# nayaLafz
### RNN & LSTM Language Modelling — Next Word Prediction

**Corpus:** Jane Austen / Charles Dickens + Tiny Shakespeare (primary) · BBC News (out-of-domain)  
**Phases:** 1 Data Engineering · 2 Vanishing Gradient · 3 Architecture Search · 4 Chatbot · 5 Confidence Analysis · 6 Style Adaptation

---

## Phase 1 — Sequential Data Engineering

Transforms raw literary text into windowed token sequences.  
**Output:** `checkpoints/phase1_data.pkl` — loaded by every later phase.

In [ ]:
import os, sys, re, random, pickle, hashlib
from pathlib import Path
from collections import Counter
import numpy as np

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print('pandas not available; CSV loading disabled')

# ── Environment detection ────────────────────────────────────────────────────
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = os.environ.get('NAYALAFZ_BASE', '/content/drive/MyDrive/nayaLafz')
elif IN_KAGGLE:
    BASE = os.environ.get('NAYALAFZ_BASE', '/kaggle/working')
else:
    BASE = os.environ.get('NAYALAFZ_BASE', str(Path.cwd().parent))

DATA_DIR = os.path.join(BASE, 'data')
CKPT_DIR = os.path.join(BASE, 'checkpoints')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# True locally → uses a small subset so the notebook runs end-to-end in minutes
QUICK_TEST = not (IN_COLAB or IN_KAGGLE)

env = 'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'
print(f'Environment : {env}')
print(f'BASE        : {BASE}')
print(f'QUICK_TEST  : {QUICK_TEST}')

In [ ]:
# ── Kaggle credentials ───────────────────────────────────────────────────────
# Colab: add KAGGLE_USERNAME and KAGGLE_KEY in the Secrets panel (key icon)
# Local / university: place kaggle.json in ~/.kaggle/

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Kaggle credentials loaded from Colab Secrets')
    except Exception as e:
        print(f'Could not load Colab Secrets: {e}')
        print('Set KAGGLE_USERNAME / KAGGLE_KEY env vars manually if download fails')
else:
    kag = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kag):
        print(f'Kaggle credentials found at {kag}')
    else:
        print('WARNING: ~/.kaggle/kaggle.json not found.')
        print('Run download_data.sh from the repo root before continuing.')

### 1.0 Dataset Inventory

Print what is actually on disk before loading anything — file names and sizes vary by Kaggle version.

In [ ]:
AUSTEN_DIR      = os.path.join(DATA_DIR, 'austen')
SHAKESPEARE_DIR = os.path.join(DATA_DIR, 'shakespeare')
BBC_DIR         = os.path.join(DATA_DIR, 'bbc')

def list_dir(label, path, max_show=20):
    if not os.path.exists(path):
        print(f'{label}: NOT FOUND at {path}')
        return []
    all_files = []
    for root, _, files in os.walk(path):
        for f in files:
            all_files.append(os.path.relpath(os.path.join(root, f), path))
    all_files.sort()
    print(f'{label} — {len(all_files)} files:')
    for fp in all_files[:max_show]:
        sz = os.path.getsize(os.path.join(path, fp))
        print(f'  {fp:<50}  {sz:>10,} B')
    if len(all_files) > max_show:
        print(f'  ... +{len(all_files) - max_show} more')
    return all_files

print('=== Dataset inventory ===\n')
austen_files      = list_dir('Austen / Dickens', AUSTEN_DIR)
print()
shakespeare_files = list_dir('Shakespeare',      SHAKESPEARE_DIR)
print()
bbc_files         = list_dir('BBC',              BBC_DIR, max_show=10)

### 1.1 Load Raw Data

Handles both `.txt` and `.csv` files, inferring the text column automatically.

In [ ]:
def load_txt_dir(directory):
    texts = []
    for root, _, files in os.walk(directory):
        for fname in sorted(files):
            if not fname.lower().endswith('.txt'):
                continue
            fp = os.path.join(root, fname)
            try:
                with open(fp, 'r', encoding='utf-8', errors='replace') as f:
                    texts.append(f.read())
            except Exception as e:
                print(f'  skipping {fname}: {e}')
    return '\n\n'.join(texts)

def load_csv_dir(directory):
    if not HAS_PANDAS:
        return ''
    texts = []
    for root, _, files in os.walk(directory):
        for fname in sorted(files):
            if not fname.lower().endswith('.csv'):
                continue
            fp = os.path.join(root, fname)
            try:
                df = pd.read_csv(fp)
                text_cols = [c for c in df.columns
                             if any(k in c.lower() for k in ('text', 'sentence', 'content', 'body'))]
                if not text_cols:
                    text_cols = df.select_dtypes(include='object').columns.tolist()
                if text_cols:
                    texts.append('\n'.join(df[text_cols[0]].dropna().astype(str).tolist()))
            except Exception as e:
                print(f'  skipping {fname}: {e}')
    return '\n'.join(texts)

def load_corpus(directory):
    txt = load_txt_dir(directory)
    csv = load_csv_dir(directory)
    return (txt + '\n' + csv).strip()

In [ ]:
print('Loading Austen / Dickens...')
austen_raw = load_corpus(AUSTEN_DIR)
print(f'  {len(austen_raw):,} chars')
print('  Preview:', austen_raw[:300].replace('\n', ' '))
print()

print('Loading Shakespeare...')
shakespeare_raw = load_corpus(SHAKESPEARE_DIR)
print(f'  {len(shakespeare_raw):,} chars')
print('  Preview:', shakespeare_raw[:300].replace('\n', ' '))

In [ ]:
def load_bbc_deduped(bbc_dir):
    seen, articles, skipped = set(), [], 0
    for root, _, files in os.walk(bbc_dir):
        for fname in sorted(files):
            if not fname.lower().endswith('.txt'):
                continue
            fp = os.path.join(root, fname)
            try:
                with open(fp, 'r', encoding='utf-8', errors='replace') as f:
                    content = f.read().strip()
            except Exception:
                continue
            h = hashlib.md5(content.encode()).hexdigest()
            if h in seen:
                skipped += 1
                continue
            seen.add(h)
            articles.append(content)
    print(f'BBC: {len(articles)} unique articles, {skipped} duplicates removed')
    return articles

bbc_articles = load_bbc_deduped(BBC_DIR)
print(f'BBC total chars: {sum(len(a) for a in bbc_articles):,}')

### 1.2 Data Cleaning

**Gutenberg boilerplate** — strip everything outside the `*** START OF` / `*** END OF` markers and remove standalone chapter headings.

In [ ]:
def clean_gutenberg(text):
    upper = text.upper()

    # Strip header
    for marker in ('*** START OF', '***START OF'):
        idx = upper.find(marker)
        if idx != -1:
            nl = text.find('\n', idx)
            if nl != -1:
                text  = text[nl + 1:]
                upper = text.upper()
            break

    # Strip footer
    for marker in ('*** END OF', '***END OF', 'END OF THE PROJECT GUTENBERG'):
        idx = upper.find(marker)
        if idx != -1:
            text  = text[:idx]
            upper = text.upper()
            break

    # Remove standalone chapter headings
    lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.upper().startswith('CHAPTER') and len(stripped.split()) <= 5:
            continue
        lines.append(line)
    text = '\n'.join(lines)

    # Collapse runs of blank lines
    while '\n\n\n' in text:
        text = text.replace('\n\n\n', '\n\n')

    return text.strip()

austen_clean = clean_gutenberg(austen_raw)
removed = len(austen_raw) - len(austen_clean)
print(f'Austen: {len(austen_raw):,} → {len(austen_clean):,} chars  ({removed:,} removed)')

shakespeare_clean = shakespeare_raw  # Tiny Shakespeare is already clean

### 1.3 Corpus Split — 80 / 10 / 10

Split by **whole sentences** with a fixed seed, **before** building the vocabulary.  
Nothing from val or test leaks into the vocab.

In [ ]:
def split_sentences(text):
    text  = ' '.join(text.split())                   # normalize whitespace
    parts = re.split(r'(?<=[.!?]) +', text)          # split on sentence-ending punct
    return [p.strip() for p in parts if len(p.strip().split()) >= 4]

def tokenize(text):
    text = text.lower()
    chars = []
    apos = chr(39)   # apostrophe — avoids quote escaping issues
    for ch in text:
        chars.append(ch if (ch.isalpha() or ch.isdigit() or ch == apos) else ' ')
    return ''.join(chars).split()

print('Splitting sentences...')
austen_sents      = split_sentences(austen_clean)
shakespeare_sents = split_sentences(shakespeare_clean)
all_sents         = austen_sents + shakespeare_sents

print(f'Austen sentences      : {len(austen_sents):,}')
print(f'Shakespeare sentences : {len(shakespeare_sents):,}')
print(f'Total                 : {len(all_sents):,}')
print()
print('Sample sentences:')
for s in random.sample(all_sents, 3):
    print(f'  {s[:100]}')

In [ ]:
random.seed(SEED)
shuffled = all_sents[:]
random.shuffle(shuffled)

n       = len(shuffled)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_sents = shuffled[:n_train]
val_sents   = shuffled[n_train : n_train + n_val]
test_sents  = shuffled[n_train + n_val :]

print(f'Train : {len(train_sents):>8,}  ({len(train_sents) / n:.0%})')
print(f'Val   : {len(val_sents):>8,}  ({len(val_sents)   / n:.0%})')
print(f'Test  : {len(test_sents):>8,}  ({len(test_sents)  / n:.0%})')

### 1.4 Vocabulary

Built from the **training split only**.  
Words with fewer than `MIN_COUNT` occurrences are mapped to `<UNK>`.

In [ ]:
MIN_COUNT = 3

PAD = '<PAD>'
UNK = '<UNK>'
BOS = '<s>'
EOS = '</s>'
SPECIAL = [PAD, UNK, BOS, EOS]

train_counts = Counter()
for sent in train_sents:
    train_counts.update(tokenize(sent))

vocab_words = sorted(w for w, c in train_counts.items() if c >= MIN_COUNT)
vocab       = SPECIAL + vocab_words
word2idx    = {w: i for i, w in enumerate(vocab)}
idx2word    = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE  = len(vocab)

print(f'MIN_COUNT      : {MIN_COUNT}')
print(f'Total tokens   : {sum(train_counts.values()):,}')
print(f'Unique types   : {len(train_counts):,}')
print(f'Vocab size     : {VOCAB_SIZE:,}  (includes {len(SPECIAL)} special tokens)')
print(f'OOV types      : {len(train_counts) - len(vocab_words):,}')

In [ ]:
def compute_unk_rate(sentences):
    total = unk_n = 0
    for sent in sentences:
        for tok in tokenize(sent):
            total += 1
            if tok not in word2idx:
                unk_n += 1
    return unk_n / total if total else 0.0

print(f'{"Split":<8} {"Sentences":>10} {"Tokens":>10} {"UNK rate":>10}')
print('-' * 42)
for name, sents in [('Train', train_sents), ('Val', val_sents), ('Test', test_sents)]:
    n_tok = sum(len(tokenize(s)) for s in sents)
    rate  = compute_unk_rate(sents)
    print(f'{name:<8} {len(sents):>10,} {n_tok:>10,} {rate:>9.2%}')

### 1.5 Sliding Window Tokenizer

Each sentence is wrapped with `<s>` / `</s>` boundary tokens, then a window of size `k` slides across to produce `(input_window, next_word)` pairs.

- Default `k = 20` for Phases 3–6  
- Phase 2 calls `make_windows` with `k ∈ {5, 10, 20, 40, 80}`

In [ ]:
DEFAULT_K = 20

def make_windows(sentences, k=DEFAULT_K, limit=None):
    if limit is not None:
        sentences = sentences[:limit]
    X, y = [], []
    for sent in sentences:
        toks    = [BOS] + tokenize(sent) + [EOS]
        indices = [word2idx.get(t, word2idx[UNK]) for t in toks]
        for i in range(len(indices) - k):
            X.append(indices[i : i + k])
            y.append(indices[i + k])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

lim = 1000 if QUICK_TEST else None
lim_small = lim // 5 if lim else None

print(f'Building windows  k={DEFAULT_K}  (QUICK_TEST={QUICK_TEST})...')
train_X, train_y = make_windows(train_sents, k=DEFAULT_K, limit=lim)
val_X,   val_y   = make_windows(val_sents,   k=DEFAULT_K, limit=lim_small)
test_X,  test_y  = make_windows(test_sents,  k=DEFAULT_K, limit=lim_small)

print(f'Train : {len(train_X):>10,} windows  shape={train_X.shape}')
print(f'Val   : {len(val_X):>10,} windows  shape={val_X.shape}')
print(f'Test  : {len(test_X):>10,} windows  shape={test_X.shape}')

### 1.6 Phase 1 Summary & Save

In [ ]:
save_path = os.path.join(CKPT_DIR, 'phase1_data.pkl')

phase1 = {
    'word2idx'     : word2idx,
    'idx2word'     : idx2word,
    'vocab_size'   : VOCAB_SIZE,
    'special'      : {'PAD': PAD, 'UNK': UNK, 'BOS': BOS, 'EOS': EOS},
    'min_count'    : MIN_COUNT,
    'k'            : DEFAULT_K,
    'seed'         : SEED,
    'train_sents'  : train_sents,
    'val_sents'    : val_sents,
    'test_sents'   : test_sents,
    'train_X'      : train_X,
    'train_y'      : train_y,
    'val_X'        : val_X,
    'val_y'        : val_y,
    'test_X'       : test_X,
    'test_y'       : test_y,
    'bbc_articles' : bbc_articles,
}

with open(save_path, 'wb') as f:
    pickle.dump(phase1, f)

mb = os.path.getsize(save_path) / 1024 ** 2
print(f'Saved: {save_path}  ({mb:.1f} MB)')
print()
print('=== Phase 1 Summary ===')
print(f'  Vocabulary size : {VOCAB_SIZE:,}')
print(f'  MIN_COUNT       : {MIN_COUNT}')
print(f'  Window size k   : {DEFAULT_K}')
print(f'  Train windows   : {len(train_X):,}')
print(f'  Val windows     : {len(val_X):,}')
print(f'  Test windows    : {len(test_X):,}')
print(f'  BBC articles    : {len(bbc_articles):,}')
print(f'  UNK rate train  : {compute_unk_rate(train_sents):.2%}')
print(f'  UNK rate val    : {compute_unk_rate(val_sents):.2%}')
print(f'  UNK rate test   : {compute_unk_rate(test_sents):.2%}')